# Common High Priority items (CHPI): extraction from the PDF list
Reads the EU list of Common High Priority items (`list-common-high-priority-items_en.pdf`) with `pdfplumber` and writes one row per HS code with its tier and description. Written for Google Colab: upload the PDF to `/content/`.

In [ ]:
!pip install pdfplumber pandas


In [ ]:
import pdfplumber
import pandas as pd
import re
from google.colab import files

pdf_filename = "/content/list-common-high-priority-items_en.pdf"

# ── Tier headers appear just above each table in the text ─────
# We extract them by reading the text of each page and finding
# TIER labels, then match them to tables in order of appearance

records      = []
current_tier = "UNKNOWN"
tier_pattern = re.compile(r"(TIER\s+[\d]+(?:\.[AB])?)", re.IGNORECASE)

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:

        # Find all tier labels on this page in order
        text        = page.extract_text() or ""
        tier_labels = tier_pattern.findall(text)
        tier_labels = [t.upper().strip() for t in tier_labels
                       if not any(x in t.upper() for x in
                                  ["COMPRISES", "WHICH", "DIVIDED",
                                   "CONTAINS", "INCLUDES", "LISTS"])]

        # Extract all tables on this page
        tables = page.extract_tables()

        # Match each table to its tier label in order
        for t_idx, table in enumerate(tables):
            if t_idx < len(tier_labels):
                current_tier = tier_labels[t_idx]

            for row in table:
                # Find the CN code: first cell matching XXXX.XX pattern
                code = None
                desc = None
                for cell in row:
                    if cell and re.match(r"^\d{4}\.\d{2}$", str(cell).strip()):
                        code = cell.strip()
                    elif cell and not re.match(
                            r"^HS code|^Text$", str(cell).strip(), re.IGNORECASE):
                        # Take the first non-None, non-header cell as description
                        if code and desc is None:
                            desc = cell.strip()

                if not code or not desc:
                    continue

                # Clean description: replace newlines with spaces
                desc       = re.sub(r"\s+", " ", desc).strip()
                normalized = code.replace(".", " ")
                digits     = len(re.sub(r"\D", "", normalized))
                cn_type    = {4: "HS4", 6: "HS6", 8: "HS8"}.get(digits, "unknown")

                records.append({
                    "tier":        current_tier,
                    "cn_code":     normalized,
                    "cn_type":     cn_type,
                    "description": desc
                })

# ── Build DataFrame ───────────────────────────────────────────
df = pd.DataFrame(records)

print(f"Total CN codes extracted: {len(df)}")
print()
print("Breakdown by tier:")
print(df.groupby("tier")["cn_code"].count().reset_index()
      .rename(columns={"cn_code": "count"}).to_string())
print()
print(df.to_string())

# ── Save and download ─────────────────────────────────────────
output_file = "CN_common_high_priority_items.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"\nSaved as: {output_file}")
files.download(output_file)